<a href="https://colab.research.google.com/github/Altaieb-Mohammed/lab_2corse/blob/master/lab2_3_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import os
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import warnings
warnings.filterwarnings('ignore')

# Установите путь к вашему датасету
data_dir = '/content/casting_data/casting_data/'
train_dir = os.path.join(data_dir, 'train')
test_dir = os.path.join(data_dir, 'test')

Шаг 1: Анализ датасета

In [2]:
def analyze_dataset():
    # Считаем количество изображений в каждой категории
    categories = ['ok_front', 'def_front']

    train_counts = {}
    test_counts = {}

    for category in categories:
        train_path = os.path.join(train_dir, category)
        test_path = os.path.join(test_dir, category)

        train_counts[category] = len(os.listdir(train_path))
        test_counts[category] = len(os.listdir(test_path))

    # Создаем DataFrame для анализа
    analysis_df = pd.DataFrame({
        'Категория': list(train_counts.keys()),
        'Тренировочные': list(train_counts.values()),
        'Тестовые': list(test_counts.values())
    })

    analysis_df['Всего'] = analysis_df['Тренировочные'] + analysis_df['Тестовые']
    analysis_df['Доля, %'] = (analysis_df['Тренировочные'] / analysis_df['Тренировочные'].sum() * 100).round(2)

    print("Анализ датасета:")
    print(analysis_df)
    print(f"\nВсего изображений: {analysis_df['Всего'].sum()}")
    print(f"Тренировочных: {analysis_df['Тренировочные'].sum()}")
    print(f"Тестовых: {analysis_df['Тестовые'].sum()}")

    # Визуализация
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Гистограмма распределения
    axes[0].bar(analysis_df['Категория'], analysis_df['Тренировочные'], alpha=0.7, label='Тренировочные')
    axes[0].bar(analysis_df['Категория'], analysis_df['Тестовые'], alpha=0.7, label='Тестовые', bottom=analysis_df['Тренировочные'])
    axes[0].set_title('Распределение изображений по категориям')
    axes[0].set_ylabel('Количество')
    axes[0].legend()

    # Круговые диаграммы
    axes[1].pie(analysis_df['Тренировочные'], labels=analysis_df['Категория'], autopct='%1.1f%%')
    axes[1].set_title('Доли классов в тренировочной выборке')

    plt.tight_layout()
    plt.show()

    return analysis_df

# Запускаем анализ
dataset_stats = analyze_dataset()

FileNotFoundError: [Errno 2] No such file or directory: '/content/casting_data/casting_data/train/ok_front'

Шаг 2: Загрузка и предобработка **изображений**


In [ ]:
def load_and_preprocess_images(image_dir, target_size=(224, 224), max_samples=None):
    """
    Загрузка и предобработка изображений
    """
    images = []
    labels = []
    filenames = []

    categories = ['ok_front', 'def_front']
    label_map = {'ok_front': 0, 'def_front': 1}

    for category in categories:
        category_path = os.path.join(image_dir, category)
        image_files = os.listdir(category_path)

        if max_samples:
            image_files = image_files[:max_samples]

        for img_file in image_files:
            img_path = os.path.join(category_path, img_file)

            try:
                # Загружаем изображение
                img = load_img(img_path, target_size=target_size)

                # Конвертируем в массив
                img_array = img_to_array(img)

                # Предобработка для MobileNetV2
                img_array = preprocess_input(img_array)

                images.append(img_array)
                labels.append(label_map[category])
                filenames.append(img_file)

            except Exception as e:
                print(f"Ошибка при загрузке {img_path}: {e}")

    return np.array(images), np.array(labels), filenames

# Загружаем тренировочные данные
print("Загрузка тренировочных изображений...")
X_train_full, y_train_full, train_filenames = load_and_preprocess_images(train_dir, max_samples=1000)  # Ограничим для скорости

print("Загрузка тестовых изображений...")
X_test, y_test, test_filenames = load_and_preprocess_images(test_dir, max_samples=200)

print(f"Форма тренировочных данных: {X_train_full.shape}")
print(f"Форма тестовых данных: {X_test.shape}")

Шаг 3: Извлечение признаков с помощью предобученной CNN

In [ ]:
def extract_features(images, batch_size=32):
    """
    Извлечение признаков с использованием предобученной MobileNetV2
    """
    # Загружаем предобученную модель без верхних слоев
    base_model = MobileNetV2(weights='imagenet', include_top=False,
                           input_shape=(224, 224, 3), pooling='avg')

    # Замораживаем веса базовой модели
    base_model.trainable = False

    # Извлекаем признаки
    features = base_model.predict(images, batch_size=batch_size, verbose=1)

    return features

print("Извлечение признаков из тренировочных изображений...")
train_features = extract_features(X_train_full)

print("Извлечение признаков из тестовых изображений...")
test_features = extract_features(X_test)

print(f"Форма извлеченных признаков (train): {train_features.shape}")
print(f"Форма извлеченных признаков (test): {test_features.shape}")

Шаг 4: Разделение на тренировочную и валидационную выборки

In [ ]:
# Разделяем тренировочные данные на тренировочную и валидационную выборки
X_train, X_val, y_train, y_val = train_test_split(
    train_features, y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full  # Сохраняем распределение классов
)

print(f"Тренировочная выборка: {X_train.shape}")
print(f"Валидационная выборка: {X_val.shape}")
print(f"Тестовая выборка: {test_features.shape}")

# Проверяем распределение классов
print("\nРаспределение классов:")
print(f"Тренировочная: {np.bincount(y_train)}")
print(f"Валидационная: {np.bincount(y_val)}")
print(f"Тестовая: {np.bincount(y_test)}")

Шаг 5: Масштабирование признаков

In [ ]:
# Масштабируем признаки
scaler = StandardScaler()

# Обучаем scaler на тренировочных данных
X_train_scaled = scaler.fit_transform(X_train)

# Применяем к валидационным и тестовым данным
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(test_features)

print(f"Масштабированные признаки - Train: {X_train_scaled.shape}")
print(f"Среднее значение признаков (первые 5): {np.mean(X_train_scaled, axis=0)[:5]}")
print(f"Стандартное отклонение (первые 5): {np.std(X_train_scaled, axis=0)[:5]}")

Шаг 6: Кросс-валидация


In [ ]:
# Создаем модель для классификации
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

# Вариант 1: Простая кросс-валидация
print("Простая кросс-валидация (5 фолдов):")
cv_scores = cross_val_score(
    model, X_train_scaled, y_train,
    cv=5,  # 5-fold cross-validation
    scoring='accuracy',
    n_jobs=-1
)

print(f"Оценки по фолдам: {cv_scores}")
print(f"Средняя точность: {cv_scores.mean():.4f}")
print(f"Стандартное отклонение: {cv_scores.std():.4f}")

# Вариант 2: Стратифицированная кросс-валидация
print("\nСтратифицированная кросс-валидация:")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

stratified_scores = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_scaled, y_train), 1):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]

    # Обучаем модель
    model.fit(X_fold_train, y_fold_train)

    # Предсказываем на валидационном фолде
    y_pred = model.predict(X_fold_val)

    # Вычисляем точность
    accuracy = accuracy_score(y_fold_val, y_pred)
    stratified_scores.append(accuracy)

    print(f"Фолд {fold}: Точность = {accuracy:.4f}")

print(f"\nИтоговая средняя точность: {np.mean(stratified_scores):.4f}")

Шаг 7: Обучение финальной модели и оценка

In [ ]:
# Обучаем модель на всех тренировочных данных
print("Обучение финальной модели...")
model.fit(X_train_scaled, y_train)

# Оценка на валидационной выборке
y_val_pred = model.predict(X_val_scaled)
val_accuracy = accuracy_score(y_val, y_val_pred)
print(f"Точность на валидационной выборке: {val_accuracy:.4f}")

# Оценка на тестовой выборке
y_test_pred = model.predict(X_test_scaled)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Точность на тестовой выборке: {test_accuracy:.4f}")

# Детальный отчет
print("\n" + "="*50)
print("КЛАССИФИКАЦИОННЫЙ ОТЧЕТ (Тестовая выборка):")
print("="*50)
print(classification_report(y_test, y_test_pred,
                           target_names=['OK (0)', 'Defect (1)']))

# Матрица ошибок
print("\nМАТРИЦА ОШИБОК:")
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['OK', 'Defect'],
            yticklabels=['OK', 'Defect'])
plt.title('Матрица ошибок')
plt.ylabel('Истинный класс')
plt.xlabel('Предсказанный класс')
plt.show()

# Важность признаков (топ-20)
if hasattr(model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': [f'feat_{i}' for i in range(len(model.feature_importances_))],
        'importance': model.feature_importances_
    })
    feature_importance = feature_importance.sort_values('importance', ascending=False)

    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance['feature'][:20][::-1],
             feature_importance['importance'][:20][::-1])
    plt.xlabel('Важность признака')
    plt.title('Топ-20 наиболее важных признаков')
    plt.tight_layout()
    plt.show()

Шаг 8: Анализ результатов

In [ ]:
def analyze_results(y_true, y_pred, filenames, data_type="тестовой"):
    """
    Анализ результатов классификации
    """
    results_df = pd.DataFrame({
        'filename': filenames,
        'true_label': y_true,
        'pred_label': y_pred,
        'correct': y_true == y_pred
    })

    # Анализ ошибок
    errors = results_df[~results_df['correct']]

    print(f"\nАнализ ошибок на {data_type} выборке:")
    print(f"Всего образцов: {len(results_df)}")
    print(f"Количество ошибок: {len(errors)}")
    print(f"Точность: {(len(results_df) - len(errors)) / len(results_df) * 100:.2f}%")

    if len(errors) > 0:
        print("\nРаспределение ошибок по классам:")
        error_counts = errors.groupby('true_label').size()
        for label, count in error_counts.items():
            class_name = "OK" if label == 0 else "Defect"
            print(f"  {class_name} ошибочно классифицирован: {count} раз")

    return results_df

# Анализ результатов на тестовой выборке
test_results = analyze_results(y_test, y_test_pred, test_filenames, "тестовой")

Дополнительно: Улучшение с помощью аугментации данных

In [ ]:
# Создаем генератор аугментированных данных
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Пример применения аугментации
def visualize_augmentation(images, labels, num_samples=5):
    """
    Визуализация аугментированных изображений
    """
    fig, axes = plt.subplots(num_samples, 5, figsize=(15, num_samples*3))

    for i in range(num_samples):
        img = images[i]
        label = labels[i]

        # Оригинальное изображение
        axes[i, 0].imshow(img / 255.0)  # Денормализуем для отображения
        axes[i, 0].set_title(f'Original (Class: {label})')
        axes[i, 0].axis('off')

        # Аугментированные версии
        img_batch = np.expand_dims(img, axis=0)
        for j in range(4):
            aug_img = datagen.flow(img_batch, batch_size=1).next()[0]
            axes[i, j+1].imshow(aug_img / 255.0)
            axes[i, j+1].set_title(f'Augmented {j+1}')
            axes[i, j+1].axis('off')

    plt.tight_layout()
    plt.show()

# Визуализируем аугментацию (используем денормализованные изображения)
X_train_denorm = X_train_full * 127.5 + 127.5  # Обратная предобработка для MobileNet
visualize_augmentation(X_train_denorm[:5], y_train_full[:5])